In [ ]:
# Import necessary libraries
import os
import time
import random
from itertools import combinations_with_replacement
import json
import pandas as pd
from dataclasses import dataclass, asdict, replace
from typing import List, Dict, Any, Tuple, Optional

In [ ]:
# Define data classes for items, boxes, and placed items
# Define config class for packing configuration

@dataclass 
class Item: 
    code: str 
    length: float 
    width: float 
    height: float 
    weight: float 
    vertical_rotation: int = 1 # 1 = Allowed, 0 = Upright only
    fragile: bool = False            # True = nothing may rest on top of this item (input field "Fragile")
    max_stack_weight: float = -1.0   # max weight allowed on top, -1 = automatic (input field "MaxStackWeight")
    
    @property 
    def volume(self) -> float: 
        return self.length * self.width * self.height

@dataclass 
class Box: 
    code: str 
    length: float 
    width: float 
    height: float 
    max_weight: float 

@dataclass 
class PlacedItem: 
    code: str 
    x: float 
    y: float 
    z: float 
    l: float 
    w: float 
    h: float 
    weight: float
    fragile: bool = False
    max_stack_weight: float = -1.0

@dataclass 
class Config: 
    bin_max_fill_check_min_item_qty: int = 6 
    bin_max_fill_pct: float = 70.0 
    bin_buffer_h: float = 6.0 # 6mm height buffer
    # What decides whether the fill cap applies:
    #   "order"  = the item count of the WHOLE ORDER (iHub spec): if the order has more than
    #              bin_max_fill_check_min_item_qty items, EVERY carton of that order is capped at bin_max_fill_pct.
    #   "carton" = the item count in each carton separately (earlier notebook versions; more permissive).
    fill_cap_basis: str = "order"

    # Carton-combination search ("refine"): after the pipeline has a plan, look for a strictly better carton
    # combination (fewer cartons, then smaller largest carton, then smaller total volume), in the same order
    # as an exact solver would, using many item orderings with the "full" engine. Every accepted plan has
    # passed all 3D rules, but a combination that is not found is NOT proven infeasible (heuristic).
    refine: bool = True
    refine_tries: int = 40            # random item orderings tried per carton combination (plus 4 fixed ones)
    refine_budget_ms: float = 120.0   # time cap per order for the search
    # How the 3D engine orders the rotations it tries at a placement point:
    #   "legacy"         = original behaviour (arbitrary set order, kept for reproducing old results)
    #   "original_first" = keep the item as supplied, then the other rotations
    #   "low_height"     = lay the item flat first (smallest height first)
    #   "tall_first"     = stand the item up first (largest height first)
    orientation_mode: str = "tall_first"

    # Backfill: before a carton is closed, try the remaining items of the SAME order in its free space
    # (same 3D rules and the same fill cap). Off = previous behaviour.
    backfill: bool = False

    # Candidate placement points used by the 3D engine:
    #   "basic" = 3 new points per placed item (fast, original behaviour)
    #   "full"  = every combination of the x / y / z edges created by placed items, so free pockets
    #             beside and above items are reachable (slower, packs tighter when the fill cap is high)
    #   "auto"  = "basic" first; only when a batch cannot be packed whole in any carton (a forced split),
    #             retry that batch with "full". Keeps the speed of "basic" on easy orders.
    ep_mode: str = "auto"

    # Order in which items are fed to the 3D engine inside one carton:
    #   "volume" = largest volume first (original behaviour)
    #   "height" = tallest first (builds flat layers; usually packs tighter with ep_mode="full")
    #   "multi"  = try volume, height, footprint, longest side; keep the first that packs everything
    #              (otherwise the one that packs the most volume in that carton). Slower than "volume".
    item_order: str = "multi"

    # ---- Physical / warehousing constraints (checked by the 3D engine for every placement) ----
    # Minimum share of an item's base that must rest on the carton floor or on items directly below it.
    # 0 = off (items may float), 1 = fully supported.
    min_support_ratio: float = 0.5
    # Fragility and load bearing are NOT used in this application, so both are off by default.
    # (Kept as options: enforce_fragility honours a "Fragile" flag in the input - nothing may rest on such an item;
    #  enforce_load_bearing limits the cumulative weight on an item to "MaxStackWeight", or auto_stack_factor x
    #  its own weight when no limit is given.)
    enforce_fragility: bool = False
    enforce_load_bearing: bool = False
    auto_stack_factor: float = 2.5

    def __post_init__(self):
        # bin_max_fill_pct is a user setting, so reject values that make no sense
        if not (0 < self.bin_max_fill_pct <= 100):
            raise ValueError(f"bin_max_fill_pct must be in (0, 100], got {self.bin_max_fill_pct}")
        if self.orientation_mode not in ("legacy", "original_first", "low_height", "tall_first"):
            raise ValueError(f"unknown orientation_mode: {self.orientation_mode}")
        if self.ep_mode not in ("basic", "full", "auto"):
            raise ValueError(f"unknown ep_mode: {self.ep_mode}")
        if self.item_order not in ("volume", "height", "multi"):
            raise ValueError(f"unknown item_order: {self.item_order}")
        if self.fill_cap_basis not in ("order", "carton"):
            raise ValueError(f"unknown fill_cap_basis: {self.fill_cap_basis}")
        if self.refine_tries < 0 or self.refine_budget_ms <= 0:
            raise ValueError("refine_tries must be >= 0 and refine_budget_ms > 0")
        if not (0.0 <= self.min_support_ratio <= 1.0):
            raise ValueError(f"min_support_ratio must be in [0, 1], got {self.min_support_ratio}")
        if self.auto_stack_factor <= 0:
            raise ValueError(f"auto_stack_factor must be > 0, got {self.auto_stack_factor}")

In [ ]:
# Define default boxes for packing

DEFAULT_BOXES = [ 
  Box("Box2", 270.0, 170.0, 115.0, 20.0), 
  Box("Box4", 340.0, 260.0, 150.0, 20.0), 
  Box("Box5", 340.0, 260.0, 235.0, 20.0), 
  Box("Box6", 340.0, 260.0, 280.0, 20.0), 
  Box("Box8", 290.0, 180.0, 280.0, 20.0), 
  Box("Box9", 440.0, 345.0, 280.0, 20.0), ]

# Function to calculate usable volume of a box considering height buffer
def get_usable_volume(box: Box, config: Config) -> float:
    usable_h = max(0.0, box.height - config.bin_buffer_h)
    return box.length * box.width * usable_h


# Fill-cap rule (the "70% rule"):
# Orders with MORE than bin_max_fill_check_min_item_qty items may only fill
# bin_max_fill_pct % of the usable volume. Smaller orders may use 100%.
# This ONE function is used by both the volumetric planner and the 3D stage,
# so tuning bin_max_fill_pct changes both layers consistently.
def get_allowed_volume(box: Box, item_qty: int, config: Config) -> float:
    usable = get_usable_volume(box, config)
    if item_qty > config.bin_max_fill_check_min_item_qty:
        return usable * (config.bin_max_fill_pct / 100.0)
    return usable

In [ ]:
# STAGE 1 - VOLUMETRIC PLANNER (Capstone 1.1)
# Cheap relaxation: uses only total volume, total weight and the fill-cap rule.
# It answers "which items can share one carton, and which need several?"
# Volume/weight are NECESSARY conditions: if the planner says a batch does not fit
# a box, no 3D arrangement can rescue it. They are not SUFFICIENT: the 3D stage
# (Cell 5) must still confirm the geometry.
#
# Returns:
#   {"status": "single_box_fit" | "multi_box_split",
#    "allocations": [{"box": Box, "assigned_items": [Item, ...]}, ...],
#    "unplannable": [Item, ...]}   # items that cannot fit even an empty largest box
def solve_v1_volumetric_planner(items: List[Item], boxes: List[Box], config: Config) -> Dict[str, Any]:
    sorted_boxes = sorted(boxes, key=lambda b: get_usable_volume(b, config))  # volume-ascending
    max_box = sorted_boxes[-1]
    sorted_items = sorted(items, key=lambda i: i.volume, reverse=True)

    # Smallest box whose weight limit and fill cap accept this batch (Best Fit)
    def best_fit_box(qty: int, weight: float, vol: float) -> Optional[Box]:
        for box in sorted_boxes:
            if weight <= box.max_weight and vol <= get_allowed_volume(box, qty, config):
                return box
        return None

    # --- STEP A: try a single box for the whole set ---
    single = best_fit_box(len(sorted_items),
                          sum(i.weight for i in sorted_items),
                          sum(i.volume for i in sorted_items))
    if single:
        return {"status": "single_box_fit",
                "allocations": [{"box": single, "assigned_items": list(sorted_items)}],
                "unplannable": []}

    # --- STEP B: greedy multi-box split (batch sized by the largest carton) ---
    remaining = list(sorted_items)
    allocations, unplannable = [], []

    while remaining:
        batch, leftover = [], []
        batch_weight = batch_vol = 0.0

        for item in remaining:
            projected_qty = len(batch) + 1
            if (batch_weight + item.weight <= max_box.max_weight and
                    batch_vol + item.volume <= get_allowed_volume(max_box, projected_qty, config)):
                batch.append(item)
                batch_weight += item.weight
                batch_vol += item.volume
            else:
                leftover.append(item)

        if not batch:
            # Even alone, the first remaining item is too heavy / too big for the largest box.
            # Set it aside so the loop always makes progress (no infinite loop).
            unplannable.append(remaining[0])
            remaining = remaining[1:]
            continue

        # Best-fit box for the FINAL batch size (not a stale loop variable)
        box = best_fit_box(len(batch), batch_weight, batch_vol) or max_box
        allocations.append({"box": box, "assigned_items": batch})
        remaining = leftover

    return {"status": "multi_box_split", "allocations": allocations, "unplannable": unplannable}

In [ ]:
# 3D EXTREME POINT (EP) SPATIAL PLACEMENT ENGINE 
# 3D Constructive Placement Engine using Extreme Points (Crainic et al., 2008)

class ExtremePointPlacementEngine: 
    
    # Initialize the placement engine with a box and configuration
    def __init__(self, box: Box, config: Config): 
        self.box = box 
        self.config = config 
        self.usable_height = max(0.0, box.height - config.bin_buffer_h) 
        self.candidate_points: List[Tuple[float, float, float]] = [(0.0, 0.0, 0.0)] 
        self.placed_items: List[PlacedItem] = [] 
        self.current_weight: float = 0.0 
        self.orientation_mode = config.orientation_mode 
        self.ep_mode = config.ep_mode 
        # "full" mode: the x / y / z edge coordinates created so far (candidate points are their combinations)
        self._xs = {0.0}
        self._ys = {0.0}
        self._zs = {0.0}

    # Generate valid orientations for an item, ordered by the configured preference.
    # The engine takes the FIRST orientation that fits at a point, so the order matters.
    def get_valid_orientations(self, item: Item) -> List[Tuple[float, float, float]]: 
        l, w, h = item.length, item.width, item.height 
        upright_only = (item.vertical_rotation == 0)   # height dimension cannot be swapped

        if self.orientation_mode == "legacy":
            # Exact original behaviour (kept so earlier results stay reproducible)
            if upright_only:
                return list(set([(l, w, h), (w, l, h)]))
            return list(set([(l, w, h), (l, h, w), (w, l, h), (w, h, l), (h, l, w), (h, w, l)]))

        # Deterministic order: as supplied first, then the other rotations (duplicates removed)
        if upright_only:
            cands = [(l, w, h), (w, l, h)]
        else:
            cands = [(l, w, h), (w, l, h), (l, h, w), (h, l, w), (w, h, l), (h, w, l)]
        ordered = list(dict.fromkeys(cands))

        if self.orientation_mode == "low_height":
            return sorted(ordered, key=lambda o: o[2])          # flat first (stable sort)
        if self.orientation_mode == "tall_first":
            return sorted(ordered, key=lambda o: -o[2])         # upright first (stable sort)
        return ordered                                           # "original_first"

    @staticmethod 
    # Check if two 3D bounding boxes intersect (collision detection)
    def intersects(box1: Tuple[float, float, float, float, float, float], 
                   box2: Tuple[float, float, float, float, float, float]) -> bool: 
        x1, y1, z1, l1, w1, h1 = box1 
        x2, y2, z2, l2, w2, h2 = box2 
        return not (x1 + l1 <= x2 or 
                    x2 + l2 <= x1 or 
                    y1 + w1 <= y2 or 
                    y2 + w2 <= y1 or 
                    z1 + h1 <= z2 or 
                    z2 + h2 <= z1) 

    # Check if a candidate point is enclosed inside an already placed item
    def is_point_inside_item(self, pt: Tuple[float, float, float]) -> bool: 
        px, py, pz = pt 
        for item in self.placed_items: 
            if (item.x <= px < item.x + item.l and 
                item.y <= py < item.y + item.w and
                  item.z <= pz < item.z + item.h): 
                return True 
        return False 

    # ---- Physical / warehousing constraints ---------------------------------------------------
    # Weight limit for what may rest on top of a placed item
    def _top_weight_limit(self, q: PlacedItem) -> float:
        if q.max_stack_weight >= 0:
            return q.max_stack_weight
        if q.fragile:
            return 0.0
        return q.weight * self.config.auto_stack_factor

    # Support, fragility and load-bearing checks for placing `item` at (x, y, z) with size l x w x h
    def _physical_ok(self, x: float, y: float, z: float, l: float, w: float, h: float, item: Item) -> bool:
        cfg = self.config

        # 0. Items already placed ABOVE this spot (possible when a gap beside/under an overhang is filled later)
        #    - a fragile item may not end up under anything
        #    - this item must be able to carry what is already above it
        top = z + h
        if cfg.enforce_fragility and item.fragile:
            for t in self.placed_items:
                if (abs(t.z - top) < 1e-5 and
                        min(x + l, t.x + t.l) - max(x, t.x) > 0 and min(y + w, t.y + t.w) - max(y, t.y) > 0):
                    return False
        if cfg.enforce_load_bearing and self.placed_items:
            if item.max_stack_weight >= 0:
                own_limit = item.max_stack_weight
            elif item.fragile:
                own_limit = 0.0
            else:
                own_limit = item.weight * cfg.auto_stack_factor
            above = sum(t.weight for t in self.placed_items
                        if t.z >= top - 1e-9 and
                        min(x + l, t.x + t.l) - max(x, t.x) > 0 and min(y + w, t.y + t.w) - max(y, t.y) > 0)
            if above > own_limit + 1e-9:
                return False

        if z <= 1e-9:
            return True                                     # resting on the carton floor

        # 1. Support + fragility: items whose top face touches this item's base
        supported = 0.0
        for p in self.placed_items:
            if abs((p.z + p.h) - z) < 1e-5:
                ox = max(0.0, min(x + l, p.x + p.l) - max(x, p.x))
                oy = max(0.0, min(y + w, p.y + p.w) - max(y, p.y))
                if ox > 0 and oy > 0:
                    if cfg.enforce_fragility and p.fragile:
                        return False                        # never stack on a fragile item
                    supported += ox * oy
        area = l * w
        if area > 0 and supported / area < cfg.min_support_ratio - 1e-9:
            return False                                    # would overhang / float

        # 2. Load bearing: every item below in this column must still carry the extra weight
        if cfg.enforce_load_bearing:
            for q in self.placed_items:
                q_top = q.z + q.h
                if q_top > z + 1e-9:
                    continue                                # not below the new item
                if (min(x + l, q.x + q.l) - max(x, q.x) <= 0 or
                        min(y + w, q.y + q.w) - max(y, q.y) <= 0):
                    continue                                # different column
                load = item.weight
                for t in self.placed_items:
                    if (t.z >= q_top - 1e-9 and
                            min(t.x + t.l, q.x + q.l) - max(t.x, q.x) > 0 and
                            min(t.y + t.w, q.y + q.w) - max(t.y, q.y) > 0):
                        load += t.weight
                if load > self._top_weight_limit(q) + 1e-9:
                    return False
        return True

    # Attempt to place an item at the best available Extreme Point
    def pack_item(self, item: Item) -> bool: 
        if self.current_weight + item.weight > self.box.max_weight: 
            return False 

        if self.ep_mode == "full":
            return self._pack_item_full(item)

        # Sort candidate points: lowest Z (bottom), lowest Y (back), lowest X (left) 
        self.candidate_points.sort(key=lambda p: (p[2], p[1], p[0])) 
        orientations = self.get_valid_orientations(item) 

        for pt in list(self.candidate_points): 
            x, y, z = pt 

            for l, w, h in orientations: 
                # 1. Carton boundary check (including 6mm height buffer) 
                if x + l > self.box.length or y + w > self.box.width or z + h > self.usable_height: 
                    continue 

                # 2. Collision check against placed items 
                candidate_bbox = (x, y, z, l, w, h) 
                if any(self.intersects(candidate_bbox, (p.x, p.y, p.z, p.l, p.w, p.h)) for p in self.placed_items): 
                    continue 

                # 3. Physical constraints (support / fragility / load bearing)
                if not self._physical_ok(x, y, z, l, w, h, item):
                    continue

                # 4. Valid placement found! Place the item 
                self.placed_items.append(PlacedItem(item.code, x, y, z, l, w, h, item.weight, item.fragile, item.max_stack_weight)) 
                self.current_weight += item.weight 

                # 5. Generate new Extreme Points from exposed faces 
                self.candidate_points.remove(pt)
                new_points = [ 
                    (x + l, y, z), 
                    (x, y + w, z), 
                    (x, y, z + h) ] 

                # Filter invalid candidate points 
                for np in new_points: 
                    nx, ny, nz = np 
                    if (nx < self.box.length and ny < self.box.width and nz < self.usable_height and not self.is_point_inside_item(np) and np not in self.candidate_points): self.candidate_points.append(np) 
                
                return True 
        return False

    # "full" mode placement: candidate points are all combinations of the edge coordinates that placed
    # items have created (minus points buried inside an item). Same bounds / collision / weight rules.
    def _pack_item_full(self, item: Item) -> bool:
        orientations = self.get_valid_orientations(item)
        self.candidate_points.sort(key=lambda p: (p[2], p[1], p[0]))

        for (x, y, z) in self.candidate_points:
            for (l, w, h) in orientations:
                if x + l > self.box.length or y + w > self.box.width or z + h > self.usable_height:
                    continue
                candidate_bbox = (x, y, z, l, w, h)
                if any(self.intersects(candidate_bbox, (p.x, p.y, p.z, p.l, p.w, p.h)) for p in self.placed_items):
                    continue

                if not self._physical_ok(x, y, z, l, w, h, item):
                    continue

                # Place the item
                self.placed_items.append(PlacedItem(item.code, x, y, z, l, w, h, item.weight, item.fragile, item.max_stack_weight))
                self.current_weight += item.weight

                # New edge coordinates (only those that are still inside the carton)
                new_x = [x + l] if (x + l < self.box.length and (x + l) not in self._xs) else []
                new_y = [y + w] if (y + w < self.box.width and (y + w) not in self._ys) else []
                new_z = [z + h] if (z + h < self.usable_height and (z + h) not in self._zs) else []
                self._xs.update(new_x); self._ys.update(new_y); self._zs.update(new_z)

                # Drop candidate points now buried inside the new item
                self.candidate_points = [p for p in self.candidate_points
                                         if not (x <= p[0] < x + l and y <= p[1] < y + w and z <= p[2] < z + h)]

                # Add every combination that involves at least one NEW coordinate
                known = set(self.candidate_points)
                fresh = []
                for px in new_x:
                    fresh.extend((px, py, pz) for py in self._ys for pz in self._zs)
                for py in new_y:
                    fresh.extend((px, py, pz) for px in self._xs for pz in self._zs)
                for pz in new_z:
                    fresh.extend((px, py, pz) for px in self._xs for py in self._ys)
                for pt in fresh:
                    if pt not in known and not self.is_point_inside_item(pt):
                        known.add(pt)
                        self.candidate_points.append(pt)
                return True
        return False

In [ ]:
# UNIFIED PIPELINE  =  Stage 1 (volumetric planner)  +  Stage 2 (3D Extreme Point check)
#
#   pool = all items that can physically fit some empty carton
#   repeat:
#     plan  = volumetric_planner(pool)          # batches + assigned best-fit box (fill cap applied)
#     for each batch:
#        try boxes from the ASSIGNED box upward (smaller boxes already failed volume/cap,
#        so 3D cannot rescue them):
#           - first box where the WHOLE batch packs in 3D  -> commit it (Best Fit)
#        if no box holds the whole batch in 3D:
#           - commit the box that packs the most volume, send the rest back to the pool
#           - stop committing this round, so the leftover can be re-planned with later batches
#     leftover items (and any not-yet-committed batches) go back into the pool and are re-planned next round
#   stop when the pool is empty, or a round packs nothing (rest = unpackable)
#
# BACKFILL (Config.backfill = True): one carton per round. After the first batch is packed, the
# remaining items of the same order are tried in that carton's free space (3D rules + fill cap)
# before it is closed; whatever is left is re-planned next round.

# 3D-pack an ordered list of items into ONE carton, enforcing the same fill cap as the planner.
def _pack_in_box(box: Box, items: List[Item], config: Config):
    engine = ExtremePointPlacementEngine(box, config)
    packed, packed_vol = [], 0.0
    for item in items:
        if packed_vol + item.volume > get_allowed_volume(box, len(packed) + 1, config):
            continue  # would break the fill cap
        if engine.pack_item(item):
            packed.append(item)
            packed_vol += item.volume
    return engine, packed


# Item orderings the 3D engine can be fed with (Config.item_order)
_ORDER_KEYS = {
    "volume":    lambda i: -i.volume,
    "height":    lambda i: (-i.height, -i.volume),
    "footprint": lambda i: -(i.length * i.width),
    "longest":   lambda i: -max(i.length, i.width, i.height),
}

# 3D-pack one carton using the configured item ordering(s). Returns (engine, packed_items).
def _pack_in_box_ordered(box: Box, items: List[Item], config: Config):
    names = {"volume": ["volume"], "height": ["height"],
             "multi": ["volume", "height", "footprint", "longest"]}[config.item_order]
    best = None
    for name in names:
        engine, packed = _pack_in_box(box, sorted(items, key=_ORDER_KEYS[name]), config)
        if len(packed) == len(items):
            return engine, packed                      # whole batch packed -> done
        if best is None or sum(i.volume for i in packed) > sum(i.volume for i in best[1]):
            best = (engine, packed)
    return best


# Can this single item physically fit an EMPTY carton (dimensions, rotation, weight)?
def _item_fits_any_box(item: Item, sorted_boxes: List[Box], config: Config) -> bool:
    return any(ExtremePointPlacementEngine(b, config).pack_item(item) for b in sorted_boxes)


# Stage 2 for one planned batch. Returns (box, engine, packed_items, mode)
#   mode = "full"    -> whole batch packed
#   mode = "partial" -> forced split: some items must return to the pool
def _pack_batch_core(batch: List[Item], assigned_box: Box, sorted_boxes: List[Box], config: Config):
    start = sorted_boxes.index(assigned_box)
    attempts = []
    for box in sorted_boxes[start:]:                     # assigned box first, then larger
        engine, packed = _pack_in_box_ordered(box, batch, config)
        if len(packed) == len(batch):
            return box, engine, packed, "full"
        attempts.append((box, engine, packed))

    # Safety net: nothing at all packed from the assigned box upward (e.g. a tall, narrow item that only
    # fits a box whose total volume is below the assigned one) -> also try the smaller boxes
    if not any(att[2] for att in attempts):
        for box in sorted_boxes[:start]:
            engine, packed = _pack_in_box_ordered(box, batch, config)
            attempts.append((box, engine, packed))

    # No carton holds the whole batch: keep the attempt that packs the most volume
    # (strict '>' means ties go to the smaller carton)
    best = attempts[0]
    for att in attempts[1:]:
        if sum(i.volume for i in att[2]) > sum(i.volume for i in best[2]):
            best = att
    return best[0], best[1], best[2], "partial"


# ep_mode "auto": pack with the fast "basic" engine first; only if the batch could not be packed whole
# in any carton (forced split), retry it with the "full" engine and keep the better result.
def _pack_batch(batch: List[Item], assigned_box: Box, sorted_boxes: List[Box], config: Config):
    if config.ep_mode != "auto":
        return _pack_batch_core(batch, assigned_box, sorted_boxes, config)
    res = _pack_batch_core(batch, assigned_box, sorted_boxes, replace(config, ep_mode="basic"))
    if res[3] == "full":
        return res
    res_full = _pack_batch_core(batch, assigned_box, sorted_boxes, replace(config, ep_mode="full"))
    if res_full[3] == "full":
        return res_full
    packed_vol = lambda r: sum(i.volume for i in r[2])
    return res_full if packed_vol(res_full) > packed_vol(res) else res


# Continue packing `items` into a carton that is already open (same 3D rules, same fill cap).
# Returns the items that were added.
def _extend_carton(engine: ExtremePointPlacementEngine, box: Box, packed: List[Item],
                   items: List[Item], config: Config) -> List[Item]:
    packed_vol = sum(i.volume for i in packed)
    added: List[Item] = []
    for item in items:
        if packed_vol + item.volume > get_allowed_volume(box, len(packed) + len(added) + 1, config):
            continue
        if engine.pack_item(item):
            added.append(item)
            packed_vol += item.volume
    return added


# ---------------------------------------------------------------------------------------------
# CARTON-COMBINATION SEARCH (Config.refine)
# Enumerates carton combinations in objective order (count, largest external volume, total external volume),
# keeps only those strictly better than the current plan, and tries to pack ALL items into each one with
# many item orderings (4 fixed + refine_tries random) on the "full" engine. First success wins.
# ---------------------------------------------------------------------------------------------
def _try_combo(items: List[Item], combo: Tuple[Box, ...], cfg: Config, tries: int, rnd: random.Random, deadline: float):
    allowed = [get_allowed_volume(b, 1, cfg) for b in combo]
    seqs = [sorted(items, key=_ORDER_KEYS[name]) for name in ("volume", "height", "footprint", "longest")]
    for _ in range(tries):
        s = items[:]
        rnd.shuffle(s)
        seqs.append(s)
    for seq in seqs:
        if time.perf_counter() > deadline:
            return None
        engines = [ExtremePointPlacementEngine(b, cfg) for b in combo]
        vols = [0.0] * len(combo)
        ok = True
        for it in seq:
            placed = False
            for idx in range(len(combo)):
                if vols[idx] + it.volume > allowed[idx]:
                    continue
                if engines[idx].pack_item(it):
                    vols[idx] += it.volume
                    placed = True
                    break
            if not placed:
                ok = False
                break
        if ok:
            return engines
    return None


def _refine_cartons(items: List[Item], base_codes: List[str], boxes: List[Box], config: Config):
    """Return (combo, engines) for a strictly better carton combination, or None."""
    t0 = time.perf_counter()
    deadline = t0 + config.refine_budget_ms / 1000.0
    cfg = replace(config, ep_mode="full")
    ext = {b.code: b.length * b.width * b.height for b in boxes}
    k0 = len(base_codes)
    L0 = max(ext[c] for c in base_codes)
    T0 = sum(ext[c] for c in base_codes)
    total_vol = sum(i.volume for i in items)
    total_w = sum(i.weight for i in items)
    rnd = random.Random(0)                      # deterministic

    for k in range(1, k0 + 1):
        combos = sorted(combinations_with_replacement(boxes, k),
                        key=lambda c: (max(ext[b.code] for b in c), sum(ext[b.code] for b in c), tuple(b.code for b in c)))
        for combo in combos:
            mx = max(ext[b.code] for b in combo)
            sm = sum(ext[b.code] for b in combo)
            if k == k0 and (mx, sm) >= (L0, T0):
                break                           # sorted: nothing better remains for this carton count
            if sum(get_allowed_volume(b, 1, cfg) for b in combo) < total_vol:
                continue                        # volume (with fill cap) cannot work
            if sum(b.max_weight for b in combo) < total_w:
                continue                        # weight cannot work
            if time.perf_counter() > deadline:
                return None
            engines = _try_combo(items, combo, cfg, config.refine_tries, rnd, deadline)
            if engines is not None:
                return combo, engines
    return None


def solve_order_best_fit(order_data: Dict[str, Any], boxes: List[Box] = DEFAULT_BOXES,
                         config: Config = Config(), return_layout: bool = False) -> Dict[str, Any]:
    start_time = time.perf_counter()
    inp = order_data.get("input", order_data)
    order_id = str(inp.get("OrderNo", inp.get("OrderId", "UNKNOWN")))
    raw_items = inp.get("Items", {}).get("ItemsList", []) if isinstance(inp.get("Items"), dict) else []

    # 1. Expand items (Quantity > 1)
    items: List[Item] = []
    for item in raw_items:
        for _ in range(item.get("Quantity", 1)):
            items.append(Item(
                code=str(item.get("Code", "ITEM")),
                length=float(item.get("Length", 0)),
                width=float(item.get("Width", 0)),
                height=float(item.get("Height", 0)),
                weight=float(item.get("Weight", 0)),
                vertical_rotation=int(item.get("VerticalRotation", 1)),
                fragile=bool(item.get("Fragile", False)),
                max_stack_weight=float(item.get("MaxStackWeight", -1.0))
            ))

    backfilled = 0   # items placed by the backfill step
    refined = False  # True when the carton-combination search replaced the pipeline's plan

    def _result(status, bins, total_items, unpacked, planner_status, planner_cartons,
                upgrades, forced, rounds, layout):
        res = {
            "order_id": order_id,
            "status": status,
            "total_cartons": len(bins),
            "cartons_used": ", ".join(b["box_code"] for b in bins) if bins else "NONE",
            "total_items": total_items,
            "unpacked_items": unpacked,
            "average_fill_pct": round(sum(b["fill_pct"] for b in bins) / len(bins), 2) if bins else 0.0,
            "carton_volume": sum(b["usable_vol"] for b in bins),   # total usable volume of cartons used (lower = cheaper packaging)
            "planner_status": planner_status,
            "planner_cartons": planner_cartons,
            "box_upgrades": upgrades,      # batches that needed a LARGER box than the planner assigned
            "forced_splits": forced,       # batches the 3D engine could not pack whole in any box
            "rounds": rounds,
            "backfilled_items": backfilled,   # items added to an already-open carton (Config.backfill)
            "refined": refined,               # plan replaced by a better carton combination (Config.refine)
            "runtime_ms": round((time.perf_counter() - start_time) * 1000, 4),
        }
        if return_layout:
            res["layout"] = layout
        return res

    if not items:
        return _result("empty", [], 0, 0, None, 0, 0, 0, 0, [])

    # Fill cap basis "order": the whole order's item count decides the cap for every carton of the order.
    # Implemented by turning it into a constant cap (qty > 0 always triggers it), so both stages stay unchanged.
    if config.fill_cap_basis == "order":
        order_pct = config.bin_max_fill_pct if len(items) > config.bin_max_fill_check_min_item_qty else 100.0
        config = replace(config, bin_max_fill_pct=order_pct, bin_max_fill_check_min_item_qty=0)

    sorted_boxes = sorted(boxes, key=lambda b: get_usable_volume(b, config))  # Box2 -> ... -> Box9 by usable volume

    # 2. Pre-screen: items that cannot fit ANY empty carton (dimension / rotation / weight)
    pool, unpackable = [], []
    for item in sorted(items, key=lambda i: i.volume, reverse=True):
        (pool if _item_fits_any_box(item, sorted_boxes, config) else unpackable).append(item)

    final_bins, layout = [], []
    planner_status, planner_cartons = None, 0
    upgrades = forced = rounds = 0

    def commit_carton(box: Box, engine: ExtremePointPlacementEngine):
        usable_vol = get_usable_volume(box, config)
        used_vol = sum(p.l * p.w * p.h for p in engine.placed_items)
        final_bins.append({"box_code": box.code, "fill_pct": round(used_vol / usable_vol * 100, 2), "usable_vol": usable_vol})
        if return_layout:
            layout.append({"box": box.code, "fill_pct": final_bins[-1]["fill_pct"],
                           "placements": [asdict(p) for p in engine.placed_items]})

    # 3. Planner -> 3D check -> feedback loop
    while pool and rounds <= len(items):
        rounds += 1
        plan = solve_v1_volumetric_planner(pool, sorted_boxes, config)
        if rounds == 1:
            planner_status = plan["status"]
            planner_cartons = len(plan["allocations"])
        unpackable.extend(plan["unplannable"])

        # ---------- BACKFILL PATH: one carton per round ----------
        if config.backfill:
            unplan_ids = {id(i) for i in plan["unplannable"]}
            pool = [i for i in pool if id(i) not in unplan_ids]
            if not plan["allocations"]:
                break
            alloc = plan["allocations"][0]              # batch with the largest items first
            batch = alloc["assigned_items"]
            box, engine, packed, mode = _pack_batch(batch, alloc["box"], sorted_boxes, config)

            if not packed:                               # nothing from this batch fits any carton
                unpackable.extend(batch)
                batch_ids = {id(i) for i in batch}
                pool = [i for i in pool if id(i) not in batch_ids]
                continue

            if mode == "full" and box is not alloc["box"]:
                upgrades += 1
            if mode == "partial":
                forced += 1

            packed_ids = {id(i) for i in packed}
            rest = [i for i in pool if id(i) not in packed_ids]          # pool stays volume-descending
            if config.item_order == "height":
                rest = sorted(rest, key=_ORDER_KEYS["height"])
            extra = _extend_carton(engine, box, packed, rest, config)    # fill the free space
            backfilled += len(extra)
            commit_carton(box, engine)

            used_ids = packed_ids | {id(i) for i in extra}
            pool = [i for i in pool if id(i) not in used_ids]
            continue

        # ---------- DEFAULT PATH: commit every batch in the plan until a forced split ----------
        next_pool, packed_any, replan = [], False, False
        for alloc in plan["allocations"]:
            batch = alloc["assigned_items"]

            # After a forced split, the leftover items must be re-planned TOGETHER with the
            # not-yet-committed batches (otherwise a tiny leftover gets its own carton).
            if replan:
                next_pool.extend(batch)
                continue

            box, engine, packed, mode = _pack_batch(batch, alloc["box"], sorted_boxes, config)

            if not packed:                       # nothing from this batch fit anywhere
                next_pool.extend(batch)
                continue

            packed_any = True
            if mode == "full" and box is not alloc["box"]:
                upgrades += 1
            if mode == "partial":
                forced += 1
                replan = True

            commit_carton(box, engine)

            # Return unpacked items to the pool (identity-based: rotation-safe, duplicate-safe)
            packed_ids = {id(i) for i in packed}
            next_pool.extend(i for i in batch if id(i) not in packed_ids)

        if not packed_any:                       # no progress -> stop, flag the rest
            unpackable.extend(next_pool)
            next_pool = []
        pool = next_pool

    unpackable.extend(pool)  # safety net (round cap reached)

    # 4. Carton-combination search: try to replace the plan by a strictly better carton combination
    if config.refine and not unpackable and final_bins:
        found = _refine_cartons(items, [b["box_code"] for b in final_bins], sorted_boxes, config)
        if found is not None:
            combo, engines = found
            final_bins.clear()
            layout.clear()
            for box, engine in zip(combo, engines):
                if engine.placed_items:
                    commit_carton(box, engine)
            refined = True

    if unpackable:
        status = "unpackable"
    elif len(final_bins) == 1:
        status = "single_box_fit"
    else:
        status = "multi_box_split"

    return _result(status, final_bins, len(items), len(unpackable),
                   planner_status, planner_cartons, upgrades, forced, rounds, layout)

In [ ]:
# Execution, Summary Metrics and Fill-% Sweep

SAMPLE_ORDERS = [
    {"OrderNo": "ORD_1001", "Items": {"ItemsList": [{"Code": "SKU_63", "Length": 140, "Width": 80, "Height": 116, "Weight": 0.285, "Quantity": 2}, {"Code": "SKU_70", "Length": 160, "Width": 80, "Height": 120, "Weight": 0.187, "Quantity": 1}]}},
    {"OrderNo": "ORD_2002", "Items": {"ItemsList": [{"Code": "HEAVY_SKU", "Length": 240, "Width": 200, "Height": 180, "Weight": 8.5, "Quantity": 3}]}}
]

def load_orders(dataset_file_path: str) -> List[Dict[str, Any]]:
    print(f"Reading dataset file: {dataset_file_path} ...")
    if os.path.exists(dataset_file_path):
        with open(dataset_file_path, "r") as f:
            raw_data = json.load(f)
        return raw_data if isinstance(raw_data, list) else raw_data.get("orders", [])
    print(f"Warning: '{dataset_file_path}' not found. Using sample dataset for demonstration.")
    return SAMPLE_ORDERS

def run_orders(orders_list: List[Dict[str, Any]], config: Config) -> pd.DataFrame:
    return pd.DataFrame([solve_order_best_fit(o, DEFAULT_BOXES, config) for o in orders_list])

def print_summary(df: pd.DataFrame, config: Config):
    disagree = (df["box_upgrades"] + df["forced_splits"]) > 0
    print("=" * 80)
    print("=== BENCHMARK ORDER RESULTS (FIRST 10 ROWS) ===")
    print("=" * 80)
    print(df[["order_id", "status", "total_cartons", "cartons_used", "total_items",
              "average_fill_pct", "planner_cartons", "box_upgrades", "forced_splits", "runtime_ms"]].head(10).to_string(index=False))
    print("\n" + "=" * 80)
    print(f"=== OVERALL SUMMARY  (fill cap = {config.bin_max_fill_pct}% when items > {config.bin_max_fill_check_min_item_qty}) ===")
    print("=" * 80)
    print(f"Total Orders Processed       : {len(df)}")
    print(f"Single-Box Fits              : {(df['status'] == 'single_box_fit').sum()} ({(df['status'] == 'single_box_fit').mean()*100:.2f}%)")
    print(f"Multi-Box Splits             : {(df['status'] == 'multi_box_split').sum()} ({(df['status'] == 'multi_box_split').mean()*100:.2f}%)")
    print(f"Unpackable Orders            : {(df['status'] == 'unpackable').sum()} ({(df['status'] == 'unpackable').mean()*100:.2f}%)")
    print(f"Avg Cartons Per Order        : {df['total_cartons'].mean():.3f}")
    print(f"Average Volumetric Fill %    : {df['average_fill_pct'].mean():.2f}%")
    print(f"Planner vs 3D Disagreement   : {disagree.sum()} orders ({disagree.mean()*100:.2f}%)  [3D upgraded the box or forced a split]")
    print(f"Average Latency Per Order    : {df['runtime_ms'].mean():.4f} ms")
    print(f"P95 Latency                  : {df['runtime_ms'].quantile(0.95):.4f} ms")
    print("\nCarton Code Frequency Distribution:")
    print(df["cartons_used"].value_counts().head(10))

# Sweep the fill cap to find where cartons/fill stop improving but 3D rejects stay rare
def sweep_fill_pct(orders_list: List[Dict[str, Any]], pcts=(50, 60, 70, 80, 90, 100)) -> pd.DataFrame:
    rows = []
    for pct in pcts:
        cfg = Config(bin_max_fill_pct=float(pct))
        df = run_orders(orders_list, cfg)
        disagree = (df["box_upgrades"] + df["forced_splits"]) > 0
        rows.append({
            "fill_cap_%": pct,
            "avg_cartons": round(df["total_cartons"].mean(), 3),
            "avg_fill_%": round(df["average_fill_pct"].mean(), 2),
            "single_box_%": round((df["status"] == "single_box_fit").mean() * 100, 2),
            "unpackable_%": round((df["status"] == "unpackable").mean() * 100, 2),
            "disagreement_%": round(disagree.mean() * 100, 2),
            "avg_latency_ms": round(df["runtime_ms"].mean(), 4),
        })
    return pd.DataFrame(rows)

# Orientation experiment: same planner + same 3D engine, only the rotation preference changes.
ORIENTATION_MODES = ("legacy", "original_first", "low_height", "tall_first")

def compare_orientation_modes(orders_list: List[Dict[str, Any]], modes=ORIENTATION_MODES,
                              base_mode: str = "legacy", base_config: Optional[Config] = None) -> pd.DataFrame:
    base_config = base_config or Config()
    frames = {}
    for m in modes:
        frames[m] = run_orders(orders_list, replace(base_config, orientation_mode=m))
    base = frames[base_mode]
    rows = []
    for m, df in frames.items():
        delta = df["total_cartons"].values - base["total_cartons"].values   # same order index in every frame
        rows.append({
            "orientation_mode": m,
            "avg_cartons": round(df["total_cartons"].mean(), 4),
            "avg_fill_%": round(df["average_fill_pct"].mean(), 2),
            "carton_volume_idx": round(df["carton_volume"].mean() / base["carton_volume"].mean() * 100, 2),  # legacy = 100, lower is better
            "single_box_%": round((df["status"] == "single_box_fit").mean() * 100, 2),
            "unpackable_%": round((df["status"] == "unpackable").mean() * 100, 2),
            "orders_fewer_cartons": int((delta < 0).sum()),    # vs base_mode
            "orders_more_cartons": int((delta > 0).sum()),     # vs base_mode
            "avg_latency_ms": round(df["runtime_ms"].mean(), 4),
        })
    return pd.DataFrame(rows)

# Backfill experiment: same everything, only Config.backfill changes. Run across fill caps because the
# cap decides how much free room a carton is allowed to use (bin_max_fill_pct is a user setting).
def compare_backfill(orders_list: List[Dict[str, Any]], caps=(50, 60, 70, 80, 90, 100),
                     base_config: Optional[Config] = None) -> pd.DataFrame:
    base_config = base_config or Config()
    rows = []
    for pct in caps:
        off = run_orders(orders_list, replace(base_config, bin_max_fill_pct=float(pct), backfill=False))
        on = run_orders(orders_list, replace(base_config, bin_max_fill_pct=float(pct), backfill=True))
        delta = on["total_cartons"].values - off["total_cartons"].values
        rows.append({
            "fill_cap_%": pct,
            "cartons_off": round(off["total_cartons"].mean(), 4),
            "cartons_on": round(on["total_cartons"].mean(), 4),
            "fill_off_%": round(off["average_fill_pct"].mean(), 2),
            "fill_on_%": round(on["average_fill_pct"].mean(), 2),
            "carton_volume_idx_on": round(on["carton_volume"].mean() / off["carton_volume"].mean() * 100, 2),  # off = 100
            "orders_fewer": int((delta < 0).sum()),
            "orders_more": int((delta > 0).sum()),
            "items_backfilled": int(on["backfilled_items"].sum()),
            "latency_off_ms": round(off["runtime_ms"].mean(), 4),
            "latency_on_ms": round(on["runtime_ms"].mean(), 4),
        })
    return pd.DataFrame(rows)

# 3D engine experiment: candidate-point strategy "basic" vs "full", across fill caps, with and without backfill.
# (bin_max_fill_pct is a user setting, so the result is reported for every cap.)
def compare_engines(orders_list: List[Dict[str, Any]], caps=(50, 60, 70, 80, 90, 100),
                    base_config: Optional[Config] = None) -> pd.DataFrame:
    base_config = base_config or Config()
    rows = []
    for pct in caps:
        for bf in (False, True):
            b = run_orders(orders_list, replace(base_config, bin_max_fill_pct=float(pct), backfill=bf, ep_mode="basic"))
            f = run_orders(orders_list, replace(base_config, bin_max_fill_pct=float(pct), backfill=bf, ep_mode="full"))
            delta = f["total_cartons"].values - b["total_cartons"].values
            rows.append({
                "fill_cap_%": pct,
                "backfill": bf,
                "cartons_basic": round(b["total_cartons"].mean(), 4),
                "cartons_full": round(f["total_cartons"].mean(), 4),
                "fill_basic_%": round(b["average_fill_pct"].mean(), 2),
                "fill_full_%": round(f["average_fill_pct"].mean(), 2),
                "carton_volume_idx_full": round(f["carton_volume"].mean() / b["carton_volume"].mean() * 100, 2),  # basic = 100
                "orders_fewer": int((delta < 0).sum()),
                "orders_more": int((delta > 0).sum()),
                "latency_basic_ms": round(b["runtime_ms"].mean(), 3),
                "latency_full_ms": round(f["runtime_ms"].mean(), 3),
                "p95_full_ms": round(f["runtime_ms"].quantile(0.95), 3),
            })
    return pd.DataFrame(rows)

# Item-order x engine experiment (baseline = ep "basic" + item_order "volume" = the previous behaviour).
def compare_item_orders(orders_list: List[Dict[str, Any]], caps=(50, 70, 100),
                        base_config: Optional[Config] = None) -> pd.DataFrame:
    base_config = base_config or Config()
    rows = []
    for pct in caps:
        base = run_orders(orders_list, replace(base_config, bin_max_fill_pct=float(pct), ep_mode="basic", item_order="volume"))
        for ep in ("basic", "full"):
            for io in ("volume", "height", "multi"):
                df = run_orders(orders_list, replace(base_config, bin_max_fill_pct=float(pct), ep_mode=ep, item_order=io))
                delta = df["total_cartons"].values - base["total_cartons"].values
                rows.append({
                    "fill_cap_%": pct, "ep_mode": ep, "item_order": io,
                    "avg_cartons": round(df["total_cartons"].mean(), 4),
                    "avg_fill_%": round(df["average_fill_pct"].mean(), 2),
                    "carton_volume_idx": round(df["carton_volume"].mean() / base["carton_volume"].mean() * 100, 2),  # baseline = 100
                    "orders_fewer": int((delta < 0).sum()),
                    "orders_more": int((delta > 0).sum()),
                    "avg_latency_ms": round(df["runtime_ms"].mean(), 3),
                    "p95_latency_ms": round(df["runtime_ms"].quantile(0.95), 3),
                })
    return pd.DataFrame(rows)

# Engine-mode sweep over fill caps (item_order / orientation come from base_config).
# Baseline = ep_mode "basic" (= 100). Shows what "full" and "auto" buy and what they cost in latency.
def sweep_engine_modes(orders_list: List[Dict[str, Any]], caps=(50, 60, 70, 75, 80, 85, 90, 95, 100),
                       base_config: Optional[Config] = None) -> pd.DataFrame:
    base_config = base_config or Config()
    rows = []
    for pct in caps:
        base = run_orders(orders_list, replace(base_config, bin_max_fill_pct=float(pct), ep_mode="basic"))
        for ep in ("basic", "full", "auto"):
            df = base if ep == "basic" else run_orders(orders_list, replace(base_config, bin_max_fill_pct=float(pct), ep_mode=ep))
            delta = df["total_cartons"].values - base["total_cartons"].values
            rows.append({
                "fill_cap_%": pct, "ep_mode": ep,
                "avg_cartons": round(df["total_cartons"].mean(), 4),
                "avg_fill_%": round(df["average_fill_pct"].mean(), 2),
                "carton_volume_idx": round(df["carton_volume"].mean() / base["carton_volume"].mean() * 100, 2),  # basic = 100
                "orders_fewer": int((delta < 0).sum()),
                "orders_more": int((delta > 0).sum()),
                "avg_latency_ms": round(df["runtime_ms"].mean(), 3),
                "p95_latency_ms": round(df["runtime_ms"].quantile(0.95), 3),
            })
    return pd.DataFrame(rows)

# Physical audit: re-measures support from the returned layouts (independent of the engine's own check).
def _support_ratio_of(p: Dict[str, Any], placements: List[Dict[str, Any]]) -> float:
    if p["z"] <= 1e-9:
        return 1.0
    area = 0.0
    for q in placements:
        if q is not p and abs((q["z"] + q["h"]) - p["z"]) < 1e-5:
            ox = max(0.0, min(p["x"] + p["l"], q["x"] + q["l"]) - max(p["x"], q["x"]))
            oy = max(0.0, min(p["y"] + p["w"], q["y"] + q["w"]) - max(p["y"], q["y"]))
            area += ox * oy
    return area / (p["l"] * p["w"])

def audit_physical(orders_list: List[Dict[str, Any]], config: Config) -> Dict[str, Any]:
    items = partial = below_min = below_half = orders_partial = 0
    for o in orders_list:
        r = solve_order_best_fit(o, DEFAULT_BOXES, config, return_layout=True)
        order_flag = False
        for carton in r["layout"]:
            for p in carton["placements"]:
                items += 1
                s = _support_ratio_of(p, carton["placements"])
                if s < 0.999:
                    partial += 1
                    order_flag = True
                if s < config.min_support_ratio - 1e-9:
                    below_min += 1            # must be 0: the engine should never allow this
                if s < 0.5:
                    below_half += 1
        orders_partial += order_flag
    return {"items": items,
            "not_fully_supported_%": round(partial / items * 100, 2),
            "below_50%_support_%": round(below_half / items * 100, 2),
            "violating_min_support_items": below_min,
            "orders_with_unsupported_item_%": round(orders_partial / len(orders_list) * 100, 2)}

# Cost of the support rule: same pipeline, only min_support_ratio changes (0 = no support check = v1.9 behaviour).
def compare_physical(orders_list: List[Dict[str, Any]], ratios=(0.0, 0.5, 0.7, 0.8, 1.0),
                     base_config: Optional[Config] = None) -> pd.DataFrame:
    base_config = base_config or Config()
    base = run_orders(orders_list, replace(base_config, min_support_ratio=0.0))
    rows = []
    for ratio in ratios:
        df = base if ratio == 0.0 else run_orders(orders_list, replace(base_config, min_support_ratio=ratio))
        delta = df["total_cartons"].values - base["total_cartons"].values
        rows.append({
            "min_support": ratio,
            "avg_cartons": round(df["total_cartons"].mean(), 4),
            "avg_fill_%": round(df["average_fill_pct"].mean(), 2),
            "carton_volume_idx": round(df["carton_volume"].mean() / base["carton_volume"].mean() * 100, 2),  # no support check = 100
            "single_box_%": round((df["status"] == "single_box_fit").mean() * 100, 2),
            "unpackable_%": round((df["status"] == "unpackable").mean() * 100, 2),
            "orders_more": int((delta > 0).sum()),
            "orders_fewer": int((delta < 0).sum()),
            "avg_latency_ms": round(df["runtime_ms"].mean(), 3),
            "p95_latency_ms": round(df["runtime_ms"].quantile(0.95), 3),
        })
    return pd.DataFrame(rows)

# Carton-combination search experiment: Config.refine off vs on.
def compare_refine(orders_list: List[Dict[str, Any]], base_config: Optional[Config] = None) -> pd.DataFrame:
    base_config = base_config or Config()
    off = run_orders(orders_list, replace(base_config, refine=False))
    on = run_orders(orders_list, replace(base_config, refine=True))
    rows = []
    for label, df in (("refine off", off), ("refine on", on)):
        delta = df["total_cartons"].values - off["total_cartons"].values
        rows.append({
            "setting": label,
            "avg_cartons": round(df["total_cartons"].mean(), 4),
            "avg_fill_%": round(df["average_fill_pct"].mean(), 2),
            "carton_volume_idx": round(df["carton_volume"].mean() / off["carton_volume"].mean() * 100, 2),  # off = 100
            "orders_replaced": int(df["refined"].sum()),
            "orders_fewer": int((delta < 0).sum()),
            "orders_more": int((delta > 0).sum()),
            "avg_latency_ms": round(df["runtime_ms"].mean(), 3),
            "p95_latency_ms": round(df["runtime_ms"].quantile(0.95), 3),
        })
    return pd.DataFrame(rows)

def run_benchmark_and_print(dataset_file_path: str = "data_samples_v2.json", run_sweep: bool = True, run_orientation: bool = True, run_backfill: bool = True, run_engines: bool = False, run_item_orders: bool = False, run_engine_sweep: bool = True, run_physical: bool = True, run_refine: bool = True):
    config = Config()
    orders_list = load_orders(dataset_file_path)
    df_results = run_orders(orders_list, config)
    print_summary(df_results, config)
    if run_sweep:
        print("\n" + "=" * 80)
        print("=== FILL-CAP SWEEP ===")
        print("=" * 80)
        print(sweep_fill_pct(orders_list).to_string(index=False))
    if run_orientation:
        print("\n" + "=" * 80)
        print("=== ORIENTATION EXPERIMENT (carton_volume_idx: legacy = 100, lower is better) ===")
        print("=" * 80)
        print(compare_orientation_modes(orders_list).to_string(index=False))
    if run_backfill:
        print("\n" + "=" * 80)
        print("=== BACKFILL EXPERIMENT (carton_volume_idx_on: backfill off = 100, lower is better) ===")
        print("=" * 80)
        print(compare_backfill(orders_list).to_string(index=False))
    if run_engines:
        print("\n" + "=" * 80)
        print("=== 3D ENGINE EXPERIMENT: basic vs full candidate points (carton_volume_idx_full: basic = 100) ===")
        print("=" * 80)
        print(compare_engines(orders_list).to_string(index=False))
    if run_item_orders:
        print("\n" + "=" * 80)
        print("=== ITEM ORDER x ENGINE (baseline: basic + volume = 100, lower is better) ===")
        print("=" * 80)
        print(compare_item_orders(orders_list).to_string(index=False))
    if run_engine_sweep:
        print("\n" + "=" * 80)
        print("=== ENGINE MODE SWEEP: basic vs full vs auto (carton_volume_idx: basic = 100) ===")
        print("=" * 80)
        print(sweep_engine_modes(orders_list).to_string(index=False))
    if run_physical:
        print("\n" + "=" * 80)
        print("=== PHYSICAL CONSTRAINTS: cost of the support rule (no support check = 100) ===")
        print("=" * 80)
        print(compare_physical(orders_list, base_config=config).to_string(index=False))
        print("\nIndependent support audit with the current config:")
        print(audit_physical(orders_list, config))
    if run_refine:
        print("\n" + "=" * 80)
        print("=== CARTON-COMBINATION SEARCH: off vs on (off = 100) ===")
        print("=" * 80)
        print(compare_refine(orders_list, base_config=config).to_string(index=False))

if __name__ == "__main__":
    run_benchmark_and_print("data_samples_v2.json")